# Estadística con R

Contraste de hipótesis, regresión e intervalos de confianza utilizando conjuntos de datos integrados de R.

No requiere descarga de datos ni instalación de paquetes: utiliza únicamente R base.

## 1. Estadística descriptiva

In [ ]:
data(mtcars)
cat("Conjunto de datos: mtcars (", nrow(mtcars), "automóviles, ", ncol(mtcars), "variables)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. Prueba t de dos muestras

¿Tienen los automóviles con transmisión manual mejor rendimiento (MPG) que los automáticos?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automático:", round(mean(auto), 1), "MPG (n =", length(auto), ")\n")
cat("Transmisión manual:   ", round(mean(manual), 1), "MPG (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusión:",
    ifelse(t_result$p.value < 0.05,
           "Rechazar H0 — los automóviles manuales tienen un MPG significativamente mayor",
           "No se rechaza H0"))

## 3. Prueba de chi-cuadrado

¿Son independientes el número de cilindros y el tipo de transmisión?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automático", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. Regresión lineal múltiple

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. Diagnóstico de la regresión

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. Intervalos de confianza

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% Confianza - Intervalos:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimación", ylab = "",
     main = "95% Confianza - Intervalos para los coeficientes")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. ANOVA de una vía

¿Difiere significativamente el MPG según el número de cilindros?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nComparaciones post hoc de Tukey HSD:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG según el número de cilindros",
        xlab = "Cilindros", ylab = "Millas por galón",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## Resumen

- **Prueba t de Welch**: En la comparación unilateral no ajustada, los automóviles con transmisión manual tienen una media de MPG superior
- **Chi-cuadrado**: La tabla de contingencia sugiere una asociación, pero las bajas frecuencias esperadas activan una advertencia de aproximación, por lo que este resultado debe interpretarse con cautela
- **Regresión**: El peso y la potencia (horsepower) son predictores negativos significativos tras el ajuste; el tipo de transmisión no resulta significativo en este modelo
- **ANOVA**: El MPG difiere significativamente entre los grupos de 4, 6 y 8 cilindros; los resultados de Tukey identifican las diferencias por pares